# Joint SID-MPC Submission Notebook

Runs the MPC using the joint learned SID model from `train_joint_sid.ipynb`.

W&B logging follows the same style as `mpc_submission.ipynb`: log `district_load` during the simulation, then KPIs and temperature plot at the end.

In [1]:
import os
os.environ["WANDB_SILENT"] = "true"

import wandb
import plotly.graph_objects as go

WANDB_ENTITY = "CityLearn-TeamB"
WANDB_PROJECT = "CityLearn"

## Locate repo and dataset

In [2]:
from pathlib import Path
import sys, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.INFO)

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

HERE = Path.cwd()
REPO_DIR = None
for p in [HERE] + list(HERE.parents):
    if (p / "citylearn").exists():
        REPO_DIR = p
        break
if REPO_DIR is None:
    raise FileNotFoundError("Could not find repo root containing citylearn/.")

sys.path.insert(0, str(REPO_DIR))
from citylearn.citylearn import CityLearnEnv

# Make sure local controller files can be imported.
for p in [HERE, REPO_DIR / "MPC" / "joint_sid_mpc", REPO_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

CLIMATE = "TX"
DATASET_DIR = REPO_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
if not DATASET_DIR.exists():
    DATASET_DIR = REPO_DIR / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
SCHEMA_PATH = DATASET_DIR / "schema.json"

SID_DIR = REPO_DIR / "System Identification" / "joint_sid"

print("REPO_DIR:", REPO_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("SID_DIR:", SID_DIR)

REPO_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1
DATASET_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood
SID_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\System Identification\joint_sid


## Load target

In [3]:
sim_start = 3624
sim_end = 4343

district_target_df = pd.read_csv(DATASET_DIR / "district_target.csv")
district_target = district_target_df["district_load_target"].values[sim_start : sim_end + 1]
T_total = sim_end - sim_start + 1

print(f"Simulation window : steps {sim_start} – {sim_end} ({T_total} hours)")
print("Target shape:", district_target.shape)

Simulation window : steps 3624 – 4343 (720 hours)
Target shape: (720,)


## Create environment and controller

In [4]:
from joint_sid_model import JointSIDModel
from joint_sid_mpc_controller_coordinated import JointSIDMPCController, JointSIDMPCConfig

def make_env(central_agent=False):
    return CityLearnEnv(schema=str(SCHEMA_PATH), root_directory=str(DATASET_DIR), central_agent=central_agent)

env = make_env(central_agent=False)
obs, _ = env.reset()

sid_model = JointSIDModel(SID_DIR)

mpc_config = JointSIDMPCConfig(
    horizon=4,
    comfort_low=22.0,
    comfort_high=26.0,
    w_track=100.0,
    w_avg_track=500.0,
    w_ramp=100.0,
    w_comfort=20.0,
    w_smooth=50.0,
    w_action=5.0,
    w_soc=2.0,
    terminal_soc_ref=0.50,
    maxiter=80,
    verbose=False,
)

agent_mpc = JointSIDMPCController(
    env=env,
    sid_model=sid_model,
    district_target=district_target,
    config=mpc_config,
)

print("Joint SID-MPC initialized")

Joint SID-MPC initialized


## Run Joint SID-MPC and log district load

In [5]:
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="MPC_Coordinated_Joint_SID",
    reinit=True,
    settings=wandb.Settings(console="off")
)

wandb.define_metric("hour")
wandb.define_metric("district_load", step_metric="hour")

MAX_RUN_STEPS = None  # set to 48 for debugging

with tqdm(total=T_total if MAX_RUN_STEPS is None else min(T_total, MAX_RUN_STEPS), desc="Joint SID-MPC", unit="step") as pbar:

    while not env.terminated:

        t_before = len(env.net_electricity_consumption)
        if MAX_RUN_STEPS is not None and t_before >= MAX_RUN_STEPS:
            break

        actions = agent_mpc.predict(obs)
        obs, _, terminated, truncated, _ = env.step(actions)
        agent_mpc.update_histories_after_step(obs, actions)

        current_load = env.net_electricity_consumption[-1]
        t = len(env.net_electricity_consumption) - 1

        wandb.log({
            "hour": t,
            "district_load": float(current_load),
        })

        pbar.update(1)

        if terminated or truncated:
            break

district_load_mpc = np.array(env.net_electricity_consumption)

mpc_building_temps = np.array([
    b.energy_simulation.indoor_dry_bulb_temperature[:len(district_load_mpc)]
    for b in env.buildings
]).T

print("district_load_mpc shape:", district_load_mpc.shape)
print("mpc_building_temps shape:", mpc_building_temps.shape)

Joint SID-MPC:   0%|          | 0/720 [00:00<?, ?step/s]

district_load_mpc shape: (720,)
mpc_building_temps shape: (720, 25)


## Log KPIs

In [6]:
from SERVER.KPIs import compute_kpis

kpis_mpc = compute_kpis(district_target[:len(district_load_mpc)], district_load_mpc, mpc_building_temps)

wandb.log({
    "NMBE [%]": float(kpis_mpc["NMBE [%]"]),
    "CV-RMSE [%]": float(kpis_mpc["CV-RMSE [%]"]),
    "Temp Comfort violation [%]": float(kpis_mpc["Temp Comfort violation [%]"]),
})

print(kpis_mpc)

{'NMBE [%]': -8.083, 'CV-RMSE [%]': 74.717, 'Temp Comfort violation [%]': 33.81}


## Log temperature plot

In [7]:
T = mpc_building_temps.shape[0]

fig = go.Figure()

fig.add_hrect(
    y0=22,
    y1=26,
    fillcolor="green",
    opacity=0.12,
    line_width=0,
    annotation_text="Comfort band [22–26 °C]",
    annotation_position="top right"
)

fig.add_hline(y=22, line_dash="dash", line_color="red", line_width=1)
fig.add_hline(y=26, line_dash="dash", line_color="red", line_width=1)

for b in range(mpc_building_temps.shape[1]):

    fig.add_trace(go.Scatter(
        x=list(range(T)),
        y=mpc_building_temps[:, b],
        mode="lines",
        line=dict(color="rgba(70,130,180,0.2)", width=1),
        showlegend=False
    ))

mean_temp = mpc_building_temps.mean(axis=1)

fig.add_trace(go.Scatter(
    x=list(range(T)),
    y=mean_temp,
    mode="lines",
    line=dict(color="steelblue", width=3),
    name="Mean temperature"
))

fig.update_layout(
    title="Joint SID-MPC - Indoor Temperature",
    xaxis_title="Hour",
    yaxis_title="Temperature [°C]"
)

wandb.log({
    "temperature_plot": wandb.Plotly(fig)
})

## Finish run

In [8]:
wandb.finish()